<a href="https://colab.research.google.com/github/dr-mushtaq/natural-language-processing-projects-python/blob/main/Sentiment_Analysis_using_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import nltk
import pandas as pd
import string
from nltk.corpus import movie_reviews, stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [4]:
# --------------------------------------------------
# Download required NLTK resources (run once)
# --------------------------------------------------
nltk.download('movie_reviews')
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
# --------------------------------------------------
# Load Dataset
# --------------------------------------------------
documents = []

for category in movie_reviews.categories():
    for fileid in movie_reviews.fileids(category):
        text = movie_reviews.raw(fileid)
        documents.append((text, category))

df = pd.DataFrame(documents, columns=['review', 'sentiment'])

# Convert labels to numeric
df['sentiment'] = df['sentiment'].map({
    'neg': 0,
    'pos': 1
})

print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (2000, 2)
                                              review  sentiment
0  plot : two teen couples go to a church party ,...          0
1  the happy bastard's quick movie review \ndamn ...          0
2  it is movies like these that make a jaded movi...          0
3   " quest for camelot " is warner bros . ' firs...          0
4  synopsis : a mentally unstable man undergoing ...          0


In [8]:
# Text Preprocessing
# --------------------------------------------------
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords and non-alphabetic tokens
    tokens = [
        word for word in tokens
        if word.isalpha() and word not in stop_words
    ]

    return " ".join(tokens)

print("Preprocessing reviews...")
df['clean_review'] = df['review'].apply(preprocess_text)

# --------------------------------------------------

Preprocessing reviews...


In [9]:
# Feature Extraction (TF-IDF)
# --------------------------------------------------
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['clean_review'])
y = df['sentiment']

# --------------------------------------------------
# Train-Test Split
# --------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
# --------------------------------------------------
# Train Logistic Regression Model
# --------------------------------------------------
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

# --------------------------------------------------
# Predictions
# --------------------------------------------------
y_pred = model.predict(X_test)

# --------------------------------------------------
# Evaluation
# --------------------------------------------------
print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# --------------------------------------------------
# Test on Custom Sentences
# --------------------------------------------------
sample_reviews = [
    "This movie was fantastic and inspiring.",
    "Worst film ever. Completely boring and waste of time."
]

sample_reviews = [preprocess_text(text) for text in sample_reviews]

sample_features = tfidf.transform(sample_reviews)

predictions = model.predict(sample_features)

for review, pred in zip(sample_reviews, predictions):
    sentiment = "Positive" if pred == 1 else "Negative"
    print(f"\nReview: {review}")
    print("Predicted Sentiment:", sentiment)


Accuracy: 0.8425

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.81      0.84       200
           1       0.82      0.87      0.85       200

    accuracy                           0.84       400
   macro avg       0.84      0.84      0.84       400
weighted avg       0.84      0.84      0.84       400


Confusion Matrix:
[[163  37]
 [ 26 174]]

Review: movie fantastic inspiring
Predicted Sentiment: Positive

Review: worst film ever completely boring waste time
Predicted Sentiment: Negative
